# 诗歌生成

# 数据处理

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers, optimizers

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    import os
    # 兼容：当前目录为 chap6_RNN 或项目根目录时都能找到 poems.txt
    for path in ('poems.txt', os.path.join('chap6_RNN', 'poems.txt')):
        if os.path.exists(path):
            break
    else:
        path = 'poems.txt'
    instances, word2id, id2word = process_dataset(path)
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [2]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, inp_ids):
        '''
        此处完成建模过程，可以参考Learn2Carry
        inp_ids: (batch, seq_len) -> logits: (batch, seq_len, v_sz)
        '''
        x = self.embed_layer(inp_ids)           # (batch, seq_len, 64)
        rnn_out = self.rnn_layer(x)             # (batch, seq_len, 128)
        logits = self.dense(rnn_out)             # (batch, seq_len, v_sz)
        return logits
    
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,]；SimpleRNNCell 的 state 为单个张量 shape(b_sz, 128)
        '''
        # 若传入的是 list（如 LSTM 的 [c,h]），取第一个与 SimpleRNNCell 兼容
        if isinstance(state, (list, tuple)):
            state = state[0]
        inp_emb = self.embed_layer(x)  # shape(b_sz, emb_sz)
        h, state = self.rnncell(inp_emb, state)  # shape(b_sz, h_sz)
        logits = self.dense(h)  # shape(b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [3]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [4]:
@tf.function(reduce_retracing=True)
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = compute_loss(logits, y, seqlen)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [5]:
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 8.821059
epoch 1 : loss 6.5894947
epoch 2 : loss 6.0669665
epoch 3 : loss 5.8617663
epoch 4 : loss 5.6719112
epoch 5 : loss 5.404763
epoch 6 : loss 5.4078193
epoch 7 : loss 5.315996
epoch 8 : loss 5.3099513
epoch 9 : loss 5.1811705


# 生成过程

In [6]:
def gen_sentence(begin_word=None):
    """生成一句诗。若指定 begin_word（如 '日','红','山'），则以该字开头生成。"""
    # SimpleRNNCell 仅有一个 state，shape (1, 128)
    state = tf.zeros((1, 128))
    if begin_word is not None and begin_word in word2id:
        cur_token = tf.constant([word2id[begin_word]], dtype=tf.int32)
        collect = [begin_word]
        max_len = 49
    else:
        cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
        collect = []
        max_len = 50
    for _ in range(max_len):
        cur_token, state = model.get_next_token(cur_token, state)
        tid = int(cur_token.numpy()[0])
        if id2word.get(tid, '') == 'eos':
            break
        collect.append(id2word.get(tid, '?'))
    return ''.join(collect)

# 随机开头生成一句
print('随机开头:', gen_sentence())

随机开头: 一年春雨满，山上一枝声。


In [7]:
# 按题目要求：以「日、红、山、夜、湖、海、月」为开头词分别生成诗歌（报告截图用）
begin_words = ['日', '红', '山', '夜', '湖', '海', '月']
for w in begin_words:
    poem = gen_sentence(w)
    print(f'以「{w}」开头: {poem}')

以「日」开头: 日暮风吹，一年春水来。
以「红」开头: 红蓉树。
以「山」开头: 山上客人归，一年无处到人间。
以「夜」开头: 夜夜悠悠，一年无处人。
以「湖」开头: 湖上客，何处到人间。
以「海」开头: 海阳园里，山中一夜风。
以「月」开头: 月悠悠悠悠悠悠。
